# 5.2 Анализ поведения адаптивной модели

В этом разделе анализируется уже не просто итоговая разница между режимами, а то, как именно ведёт себя адаптивная модель.

Зачем нужен этот анализ:
- показать, какие решения принимает модель;
- проверить, реагирует ли она на ухудшение или улучшение состояния игрока;
- понять, как меняются темп и персональные смещения сложности;
- показать типичные траектории адаптации внутри сессии.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from analytics import notebook_utils as nu

In [ ]:
PREPARED_DIR = Path("analytics/data")

task_df = pd.read_csv(PREPARED_DIR / "task_table.csv")
adaptation_df = pd.read_csv(PREPARED_DIR / "adaptation_table.csv")
session_df = pd.read_csv(PREPARED_DIR / "session_table.csv")

## 1. Есть ли у нас шаги адаптации и в каких режимах

In [ ]:
adaptation_df["mode"].value_counts(dropna=False)

## 2. Какие действия модель принимает чаще всего

Это базовый анализ распределения действий адаптации.
Он показывает, склонна ли модель чаще ускорять темп, замедлять его или оставлять без изменений.

In [ ]:
adaptation_df["delta_tempo"].value_counts(dropna=False).sort_index()

In [ ]:
adaptation_df["delta_tempo"].value_counts(dropna=False).sort_index().plot(kind="bar", color="#2f7f5f")
plt.title("Распределение delta_tempo")
plt.xlabel("delta_tempo")
plt.tight_layout()

## 3. Как меняются task offsets

Этот блок нужен, чтобы показать, использует ли модель персональные смещения сложности по отдельным мини-играм.

In [ ]:
offset_cols = [
    "task_offset_compare_codes",
    "task_offset_sequence_memory",
    "task_offset_rule_switch",
    "task_offset_parity_check",
    "task_offset_radar_scan",
]

adaptation_df[offset_cols].describe()

## 4. Реагирует ли модель на состояние игрока

Здесь мы смотрим, какие действия модель принимает при разной точности и разном времени ответа перед адаптацией.

In [ ]:
adaptation_df.groupby("delta_tempo")[["state_accuracy", "state_mean_rt", "state_error_streak", "state_fatigue_trend"]].mean()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
adaptation_df.boxplot(column="state_accuracy", by="delta_tempo", ax=axes[0])
adaptation_df.boxplot(column="state_mean_rt", by="delta_tempo", ax=axes[1])
axes[0].set_title("Точность перед действием модели")
axes[1].set_title("Средний RT перед действием модели")
plt.suptitle("")
plt.tight_layout()

## 5. Что происходит после разных действий модели

Этот блок показывает, связаны ли действия модели с последующим улучшением или ухудшением состояния.

In [ ]:
next_state_view = adaptation_df[["session_id", "step", "delta_tempo", "reward", "state_accuracy", "state_mean_rt"]].copy()
next_state_view["next_accuracy"] = next_state_view.groupby("session_id")["state_accuracy"].shift(-1)
next_state_view["next_mean_rt"] = next_state_view.groupby("session_id")["state_mean_rt"].shift(-1)
next_state_view.groupby("delta_tempo")[["reward", "next_accuracy", "next_mean_rt"]].mean()

## 6. Поведение модели по ходу сессии

Здесь можно проверить, меняется ли поведение модели на ранних и поздних этапах сессии.

In [ ]:
adaptation_df["step_bucket"] = pd.cut(
    adaptation_df["step"],
    bins=[-1, 2, 5, 1000],
    labels=["start", "middle", "late"],
)

adaptation_df.groupby(["step_bucket", "delta_tempo"]).size().unstack(fill_value=0)

## 7. Типичные траектории внутри отдельных сессий

Этот блок очень полезен для иллюстраций в тексте.
Ниже можно выбрать несколько сессий и показать, как в них менялись точность, RT и tempo offset.

In [ ]:
candidate_sessions = adaptation_df["session_id"].dropna().unique().tolist()[:3]
candidate_sessions

In [ ]:
for session_id in candidate_sessions:
    one = adaptation_df[adaptation_df["session_id"] == session_id].sort_values("step")
    if one.empty:
        continue
    fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
    axes[0].plot(one["step"], one["state_accuracy"], marker="o")
    axes[0].set_title(f"{session_id}: accuracy")
    axes[1].plot(one["step"], one["state_mean_rt"], marker="o")
    axes[1].set_title(f"{session_id}: mean RT")
    axes[2].plot(one["step"], one["tempo_offset"], marker="o")
    axes[2].set_title(f"{session_id}: tempo offset")
    for ax in axes:
        ax.set_xlabel("step")
    plt.tight_layout()
    plt.show()

## 8. Анализ поведения модели по возрасту и гендеру

Если данных достаточно, можно проверить, одинаково ли ведёт себя модель в разных группах.

In [ ]:
adaptation_df.groupby(["age_group", "delta_tempo"]).size().unstack(fill_value=0)

In [ ]:
adaptation_df.groupby(["gender", "delta_tempo"]).size().unstack(fill_value=0)